In [24]:
"""
Week 4 — Baseline Model (Linear Regression)

1. Chronological train/validation/test split (Best Practices Sec. 01),
   replacing the ad-hoc constant/low-frequency/correlation-based column
   dropping that was patched onto the original script. That patch was a
   symptom fix for a design-matrix stability problem that was really coming
   from (a) naive one-hot encoding of a high-cardinality "City" column and
   (b) fitting the encoder/scaler/imputer on data the model shouldn't have
   seen. Fixing the root cause removes the need for the patch entirely.

2. All fit-time preprocessing (imputation, encoding, scaling, outlier
   thresholds) is wrapped in a scikit-learn Pipeline/ColumnTransformer and
   fit on the training window only (Sec. 07), then applied unchanged to
   validation/test — no leakage.

3. The high-cardinality "City" feature uses K-fold, CV-safe target encoding
   (Sec. 05) instead of one-hot encoding, so a category's encoding never
   sees its own row's label, and the feature space doesn't blow up into
   hundreds of sparse dummy columns (the root cause of the multicollinearity
   warnings).

4. The training-window length X is treated as tunable (per the assignment
   instructions): several candidate values are evaluated on the validation
   month, and the best one is selected — never tuned on the test month.

5. R² is reported, plus a rolling-origin
   backtest at two additional historical cutoffs to check that performance
   is stable rather than a one-off lucky split (Sec. 01 / rubric item 2).

6. Random seeds are fixed throughout for reproducibility (Sec. 11).
"""

'\nWeek 4 — Baseline Model (Linear Regression)\n\n1. Chronological train/validation/test split (Best Practices Sec. 01),\n   replacing the ad-hoc constant/low-frequency/correlation-based column\n   dropping that was patched onto the original script. That patch was a\n   symptom fix for a design-matrix stability problem that was really coming\n   from (a) naive one-hot encoding of a high-cardinality "City" column and\n   (b) fitting the encoder/scaler/imputer on data the model shouldn\'t have\n   seen. Fixing the root cause removes the need for the patch entirely.\n\n2. All fit-time preprocessing (imputation, encoding, scaling, outlier\n   thresholds) is wrapped in a scikit-learn Pipeline/ColumnTransformer and\n   fit on the training window only (Sec. 07), then applied unchanged to\n   validation/test — no leakage.\n\n3. The high-cardinality "City" feature uses K-fold, CV-safe target encoding\n   (Sec. 05) instead of one-hot encoding, so a category\'s encoding never\n   sees its own row

In [25]:
import numpy as np
import pandas as pd
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [26]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [27]:
TARGET = "ClosePrice"
HIGH_CARD_CAT = ["City"]
LOW_CARD_CAT = [
    "PropertySubType",
    "Levels",
    "AttachedGarageYN",
    "PoolPrivateYN",
    "FireplaceYN",
    "NewConstructionYN",
]

In [28]:
# ---------------------------------------------------------------------------
# Load cleaned data 
# ---------------------------------------------------------------------------
df = pd.read_csv("cleaned_single_family_sales_preprocessed.csv", parse_dates=["CloseDate"])

In [29]:
# Defensive re-normalization: 02_preprocessing.py already casts categorical
# columns to a consistent string type, but this guards against the same
# bool/str mixed-type problem re-appearing if this CSV is ever regenerated,
# hand-edited, or produced by a different upstream run. A column with real
# Python bools mixed with NaN (common for YN fields) round-trips through
# CSV as object dtype holding actual bool + float values; once SimpleImputer
# fills the NaN with the string "Unknown", OneHotEncoder is handed a mix of
# bool and str and can't sort unique categories to fit.
for col in HIGH_CARD_CAT + LOW_CARD_CAT:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str))

In [30]:
NUMERIC_COLS = [
    c for c in df.columns
    if c not in [TARGET, "CloseDate"] + HIGH_CARD_CAT + LOW_CARD_CAT
]

In [31]:
def month_period(series):
    return series.dt.to_period("M")

In [32]:
def get_window(data, end_month_exclusive, n_months):
    """Rows with CloseDate in [end_month_exclusive - n_months, end_month_exclusive)."""
    start = end_month_exclusive - n_months
    periods = month_period(data["CloseDate"])
    return data[(periods >= start) & (periods < end_month_exclusive)]

In [33]:
def get_month(data, month):
    return data[month_period(data["CloseDate"]) == month]

In [34]:
def kfold_target_encode(train_series, train_target, apply_series_list,
                         n_splits=5, smoothing=20, seed=RANDOM_SEED):
    """CV-safe target encoding (Best Practices Sec. 05): out-of-fold means
    for the training rows themselves (so a row's own label never leaks into
    its own encoding), and a single full-training-set mapping applied to
    validation/test rows."""
    train_series = train_series.reset_index(drop=True)
    train_target = train_target.reset_index(drop=True)
    global_mean = train_target.mean()

    encoded_train = np.zeros(len(train_series))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fit_idx, hold_idx in kf.split(train_series):
        fit_vals, fit_target = train_series.iloc[fit_idx], train_target.iloc[fit_idx]
        stats = fit_target.groupby(fit_vals).agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
        encoded_train[hold_idx] = train_series.iloc[hold_idx].map(smoothed).fillna(global_mean).to_numpy()

    full_stats = train_target.groupby(train_series).agg(["mean", "count"])
    full_smoothed = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
    encoded_apply = [s.map(full_smoothed).fillna(global_mean).to_numpy() for s in apply_series_list]
    return encoded_train, encoded_apply

In [35]:
def compute_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
    }

In [36]:
def fit_and_evaluate(train_df, eval_df):
    """Fits the full leak-safe pipeline on train_df only, evaluates on eval_df."""
    train_df = train_df.copy()
    eval_df = eval_df.copy()

    # Outlier thresholds learned from TRAINING data only (Sec. 03), then the
    # identical frozen cutoffs are applied to the evaluation set.
    lo, hi = train_df[TARGET].quantile([0.005, 0.995])
    train_df = train_df[(train_df[TARGET] >= lo) & (train_df[TARGET] <= hi)]
    eval_df = eval_df[(eval_df[TARGET] >= lo) & (eval_df[TARGET] <= hi)]

    y_train = train_df[TARGET].reset_index(drop=True)
    y_eval = eval_df[TARGET].reset_index(drop=True)

    # CV-safe target encoding for the high-cardinality City column.
    city_train_enc, (city_eval_enc,) = kfold_target_encode(
        train_df["City"], y_train, [eval_df["City"]]
    )

    X_train = train_df.drop(columns=[TARGET, "CloseDate", "City"]).reset_index(drop=True)
    X_train["City_target_enc"] = city_train_enc
    X_eval = eval_df.drop(columns=[TARGET, "CloseDate", "City"]).reset_index(drop=True)
    X_eval["City_target_enc"] = city_eval_enc

    numeric_final = NUMERIC_COLS + ["City_target_enc"]

    # Structural leakage prevention (Sec. 07): imputers/encoders/scalers are
    # fit on X_train only via Pipeline.fit, and merely applied (transform)
    # to X_eval.
    preprocessor = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_final),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            # min_frequency buckets rare categories instead of manually
            # dropping "low-frequency dummy columns" after the fact;
            # drop="first" avoids the dummy-variable trap that was
            # contributing to the multicollinearity warnings.
            ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", min_frequency=10)),
        ]), LOW_CARD_CAT),
    ])

    model = Pipeline([
        ("preprocess", preprocessor),
        ("regressor", LinearRegression()),
    ])

    # A rare LOW_CARD_CAT category that shows up for the first time in the
    # evaluation window (e.g. a new "Levels" value that never occurred during
    # the training window) is expected, not a bug: handle_unknown="ignore"
    # correctly encodes it as all-zeros so the model can still score the row.
    # sklearn prints a UserWarning every time this happens, which would
    # otherwise spam the console once per unseen category per window. The
    # suppression below is scoped narrowly to that exact, understood message
    # around the fit/predict calls only — any other warning still surfaces.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Found unknown categories.*",
            category=UserWarning,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_eval)

    metrics = compute_metrics(y_eval, y_pred)
    return model, metrics, y_eval, y_pred

In [37]:
# ---------------------------------------------------------------------------
# Chronological split (Sec. 01): test = last complete month, validation =
# second-to-last month, training = X months immediately before validation.
# ---------------------------------------------------------------------------
months_sorted = sorted(month_period(df["CloseDate"]).unique())
if len(months_sorted) < 3:
    raise ValueError("Need at least 3 distinct months for a train/validation/test split.")

In [38]:
test_month = months_sorted[-1]
val_month = months_sorted[-2]

In [39]:
# ---------------------------------------------------------------------------
# Tune the training-window length X on the VALIDATION month only
# (never on the test month).
# ---------------------------------------------------------------------------
candidate_X = [3, 6, 9, 12, 18, 24]
print(f"Validation month: {val_month}   Test month: {test_month}\n")
print("Tuning training-window length X on the validation month:")

Validation month: 2026-04   Test month: 2026-05

Tuning training-window length X on the validation month:


In [40]:
val_results = {}
for x in candidate_X:
    train_window = get_window(df, val_month, x)
    val_set = get_month(df, val_month)
    if len(train_window) < 100 or len(val_set) < 20:
        print(f"  X={x:>2} months -> skipped (not enough data: "
              f"train={len(train_window)}, val={len(val_set)})")
        continue
    _, metrics, _, _ = fit_and_evaluate(train_window, val_set)
    val_results[x] = metrics
    print(f"  X={x:>2} months -> R2={metrics['R2']:.4f}  ")

  X= 3 months -> R2=0.7263  
  X= 6 months -> R2=0.7528  
  X= 9 months -> R2=0.7607  
  X=12 months -> R2=0.7663  
  X=18 months -> R2=0.7689  
  X=24 months -> R2=0.7705  


In [41]:
if not val_results:
    raise ValueError("No candidate training window had enough data to evaluate.")

In [42]:
best_X = max(val_results, key=lambda x: val_results[x]["R2"])
print(f"\nSelected training window: X = {best_X} months (highest validation R²)\n")


Selected training window: X = 24 months (highest validation R²)



In [43]:
# ---------------------------------------------------------------------------
# Final model: retrain with the selected X on the window ending right before
# the test month (this naturally folds the former validation month back into
# training), then touch the test month exactly once.
# ---------------------------------------------------------------------------
final_train = get_window(df, test_month, best_X)
test_set = get_month(df, test_month)
final_model, test_metrics, y_test_actual, y_test_pred = fit_and_evaluate(final_train, test_set)

In [44]:
print("=== Final Test-Set Metric (test month touched once) ===")
print(f"R²:     {test_metrics['R2']:.4f}")

=== Final Test-Set Metric (test month touched once) ===
R²:     0.7671


In [45]:
# ---------------------------------------------------------------------------
# Rolling-origin backtest (Sec. 01 / rubric item 2): re-run the same X and
# pipeline at 2 earlier historical cutoffs to check that accuracy is stable,
# not a one-off lucky split.
# ---------------------------------------------------------------------------
print("\n=== Rolling-Origin Backtest (stability check) ===")
for offset in (1, 2):
    cutoff = test_month - offset
    if cutoff not in months_sorted:
        continue
    bt_train = get_window(df, cutoff, best_X)
    bt_eval = get_month(df, cutoff)
    if len(bt_train) < 100 or len(bt_eval) < 20:
        print(f"Cutoff {cutoff}: skipped (not enough data)")
        continue
    _, bt_metrics, _, _ = fit_and_evaluate(bt_train, bt_eval)
    print(f"Cutoff {cutoff}: R2={bt_metrics['R2']:.4f}  ")


=== Rolling-Origin Backtest (stability check) ===
Cutoff 2026-04: R2=0.7705  
Cutoff 2026-03: R2=0.7603  


In [46]:
# ---------------------------------------------------------------------------
# Save predictions
# ---------------------------------------------------------------------------
results = pd.DataFrame({"ActualPrice": y_test_actual, "PredictedPrice": y_test_pred})
results.to_csv("baseline_predictions.csv", index=False)
print("\nSaved predictions to baseline_predictions.csv")


Saved predictions to baseline_predictions.csv
